# Fine-Tuning de BETO para Clasificación de Ocupaciones (CIUO-08)

Este notebook realiza el **fine-tuning** (ajuste fino) de **BETO** —el modelo BERT entrenado en español por la Universidad de Chile— para clasificar automáticamente glosas abiertas de ocupación según la **Clasificación Internacional Uniforme de Ocupaciones** (CIUO-08) a **1 dígito** (grandes grupos).

El enfoque utiliza la API de alto nivel `Trainer` de HuggingFace, que encapsula el bucle de entrenamiento completo y reduce el código necesario al mínimo.

## 1. Importación de librerías

Importamos las herramientas principales:

- **`pandas` / `numpy`**: manipulación de datos tabulares y operaciones numéricas.
- **`sklearn`**: split train/test, codificación de etiquetas y métricas de evaluación.
- **`transformers`**: acceso al tokenizador, al modelo preentrenado y al `Trainer`.
- **`datasets`**: la clase `Dataset` de HuggingFace, que el `Trainer` puede consumir directamente sin necesidad de definir clases propias de PyTorch.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

/home/jovyan/work/conda_envs/capacitacion/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Carga de datos

Cargamos el CSV con las glosas auditadas. El archivo debe tener al menos dos columnas:

| Columna | Descripción |
|---|---|
| `texto` | Glosa de ocupación y tareas, ya concatenadas |
| `codigo_ciuo_1d` | Gran grupo CIUO-08 asignado por el auditor (0–9) |

Eliminamos filas con valores nulos en cualquier columna para evitar errores durante la tokenización.

In [2]:
df = pd.read_csv("codigos_ciuo_auditorias_nomenclaturas.csv")
df = df[["texto", "codigo_ciuo_1d"]].dropna().reset_index(drop=True)

print(f"Total de registros: {len(df):,}")
print(f"\nDistribución por gran grupo CIUO-08:")
print(df["codigo_ciuo_1d"].value_counts().sort_index())

df.head()

Total de registros: 10,195

Distribución por gran grupo CIUO-08:
codigo_ciuo_1d
-99       22
 0       181
 1       646
 2      1689
 3      1097
 4       649
 5      1508
 6       588
 7      1261
 8       701
 9      1164
 999     689
Name: count, dtype: int64


,texto,codigo_ciuo_1d
0,OCUPACIÓN: VENDER EN LA CALLE; TAREA: VENDERN,9
1,"OCUPACIÓN: CRÍA Y VENDE VACUNO.; TAREA: CRÍA, ...",6
2,OCUPACIÓN: AGRICULTOR DE TOMATES Y UVAS; TAREA...,6
3,OCUPACIÓN: COMPRA Y VENTA DE ROPA USADA; TAREA...,5
4,OCUPACIÓN: ASISTENTE COMERCIAL; TAREA: MERCADO...,3


## 3. Codificación de etiquetas

PyTorch requiere que las etiquetas sean **enteros contiguos desde 0** (0, 1, 2, …). Los códigos CIUO-08 son números del 0 al 9, pero no necesariamente están todos presentes en los datos.

`LabelEncoder` de scikit-learn hace este mapeo automáticamente: ordena las clases presentes y les asigna índices 0, 1, 2, … El objeto `le` guarda el mapeo inverso y lo usaremos al final para mostrar los códigos CIUO originales en el reporte.

In [3]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["codigo_ciuo_1d"])

mapeo = pd.DataFrame({"codigo_ciuo_1d": le.classes_, "indice_interno": range(len(le.classes_))})
print("Mapeo de etiquetas:")
print(mapeo.to_string(index=False))

Mapeo de etiquetas:
 codigo_ciuo_1d  indice_interno
            -99               0
              0               1
              1               2
              2               3
              3               4
              4               5
              5               6
              6               7
              7               8
              8               9
              9              10
            999              11


## 4. División train / test

Separamos el 20% de los datos para evaluar el modelo una vez entrenado. Este conjunto de **test** no participa en el entrenamiento: es la medida honesta del desempeño del modelo frente a glosas que nunca ha visto.

Usamos `stratify=df["label"]` para garantizar que la proporción de cada gran grupo CIUO-08 sea la misma en train y test, lo cual es importante cuando hay clases con pocos ejemplos.

In [4]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]   # mantiene proporciones por clase
)

print(f"Registros de entrenamiento: {len(train_df):,}  (80%)")
print(f"Registros de test:          {len(test_df):,}  (20%)")

Registros de entrenamiento: 8,156  (80%)
Registros de test:          2,039  (20%)


## 5. Tokenización

### ¿Qué hace el tokenizador?

Los modelos transformer no procesan texto directamente: convierten cada palabra (o subpalabra) en un número —su **token ID**— según un vocabulario fijo. El tokenizador de BETO:

1. Divide el texto en unidades subléxicas (*WordPiece*).
2. Agrega tokens especiales: `[CLS]` al inicio y `[SEP]` al final.
3. Aplica **padding** para que todos los textos tengan la misma longitud.
4. Trunca los textos que superan `max_length`.

`Dataset.from_pandas()` convierte el DataFrame directamente en un objeto que el `Trainer` puede consumir. El método `.map(tokenize, batched=True)` aplica el tokenizador sobre lotes completos de manera eficiente. Al final, `.set_format("torch")` convierte las columnas relevantes a tensores de PyTorch automáticamente.


In [5]:
MODEL = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tokenizer(
        batch["texto"],
        truncation=True,
        padding="max_length",  # padding fijo: todos los tensores quedan del mismo tamaño
        max_length=64          # suficiente para glosas cortas de ocupación
    )

ds_train = Dataset.from_pandas(train_df[["texto", "label"]].reset_index(drop=True))
ds_test  = Dataset.from_pandas(test_df[["texto", "label"]].reset_index(drop=True))

# Aplicar tokenización en batch (más rápido que ejemplo a ejemplo)
ds_train = ds_train.map(tokenize, batched=True)
ds_test  = ds_test.map(tokenize, batched=True)

# Convertir a tensores PyTorch (solo las columnas que necesita el modelo)
ds_train.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
ds_test.set_format(type="torch",  columns=["input_ids", "attention_mask", "label"])

print("Datasets listos.")
print(f"  Entrenamiento: {len(ds_train):,} ejemplos")
print(f"  Test:          {len(ds_test):,} ejemplos")

Map: 100%|██████████| 2039/2039 [00:00<00:00, 15860.91 examples/s]

Datasets listos.
  Entrenamiento: 8,156 ejemplos
  Test:          2,039 ejemplos


## 6. Carga del modelo

`AutoModelForSequenceClassification` carga BETO y le agrega automáticamente una **cabeza de clasificación**: una capa lineal de 768 → `num_labels` que produce un puntaje (logit) por clase.

Al cargar, el log muestra:
- Pesos `UNEXPECTED`: corresponden a la tarea de preentrenamiento (predicción de palabras enmascaradas), que se descartan porque no se necesitan para clasificación.
- Pesos `MISSING`: la cabeza de clasificación, que se inicializa aleatoriamente y se aprende durante el fine-tuning.

Ambos mensajes son completamente normales en este flujo.

In [6]:
num_labels = df["label"].nunique()
print(f"Número de clases (grandes grupos CIUO-08): {num_labels}")

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=num_labels)

Número de clases (grandes grupos CIUO-08): 12


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 22305.91it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from differe

## 7. Entrenamiento (Fine-Tuning)

### `compute_metrics`

Por defecto, `Trainer` solo reporta la **pérdida** (*loss*) de entrenamiento. Para obtener también exactitud y F1-score durante el entrenamiento hay que definir una función `compute_metrics` que recibe las predicciones del modelo y devuelve un diccionario de métricas.

`Trainer` llama a esta función automáticamente al final de cada época, sobre el conjunto de evaluación (`eval_dataset`).

### `TrainingArguments`

| Parámetro | Valor | Significado |
|---|---|---|
| `num_train_epochs` | 3 | Pasadas completas sobre los datos de entrenamiento |
| `per_device_train_batch_size` | 32 | Ejemplos por paso de actualización |
| `learning_rate` | 2e-5 | Tasa de aprendizaje (estándar para fine-tuning BERT) |
| `eval_strategy` | `"epoch"` | Evaluar al final de cada época |
| `logging_strategy` | `"epoch"` | Reportar métricas al final de cada época |

Con `eval_strategy="epoch"` y `eval_dataset=ds_test`, al final de cada época el `Trainer` pasa el conjunto de test por el modelo, calcula los logits sin gradientes (equivalente a `torch.no_grad()`), y llama a `compute_metrics` para mostrar las métricas en la tabla de progreso.

In [7]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": round(accuracy_score(labels, preds), 4),
        "f1_macro": round(f1_score(labels, preds, average="macro",    zero_division=0), 4),
        "f1_weighted": round(f1_score(labels, preds, average="weighted", zero_division=0), 4),
    }

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./modelo_ciuo08",
        num_train_epochs=3,
        per_device_train_batch_size=32,
        learning_rate=2e-5,
        eval_strategy="epoch",        # evaluar al final de cada época
        logging_strategy="epoch",     # reportar métricas al final de cada época
        report_to="none",
        save_strategy="no",
    ),
    train_dataset=ds_train,
    eval_dataset=ds_test,             # conjunto sobre el que se calculan las métricas
    compute_metrics=compute_metrics,
)

trainer.train()

/home/jovyan/work/conda_envs/capacitacion/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,1.640020,1.175591,0.619400,0.527200,0.609700
2,1.013557,1.031144,0.679700,0.605800,0.679900
3,0.801274,0.958786,0.698900,0.621500,0.696900


/home/jovyan/work/conda_envs/capacitacion/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/jovyan/work/conda_envs/capacitacion/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=765, training_loss=1.1516173319099776, metrics={'train_runtime': 27211.7874, 'train_samples_per_second': 0.899, 'train_steps_per_second': 0.028, 'total_flos': 804797415843840.0, 'train_loss': 1.1516173319099776, 'epoch': 3.0})

## 8. Evaluación final

La tabla de arriba ya muestra la evolución de las métricas por época. Acá generamos adicionalmente el **reporte detallado por clase** sobre el conjunto de test completo, una vez terminado el entrenamiento.

- **Precisión**: de los que el modelo predijo como clase X, ¿qué fracción realmente lo era?
- **Recall**: de los que realmente son clase X, ¿qué fracción el modelo identificó correctamente?
- **F1-score**: media armónica de precisión y recall (útil cuando las clases están desbalanceadas).
- **N**: número de ejemplos reales de esa clase en el conjunto de test.
- **Macro (promedio)**: promedio simple entre clases — cada clase pesa igual, sin importar su tamaño.
- **Weighted (promedio)**: promedio ponderado por el tamaño de cada clase.

In [8]:
# Obtener predicciones finales sobre el conjunto de test
salida = trainer.predict(ds_test)
preds  = np.argmax(salida.predictions, axis=-1)

# Etiquetas verdaderas (desde el dataset HuggingFace, con índice alineado)
etiquetas_reales = np.array(ds_test["label"])

# ── Tabla de métricas por gran grupo CIUO-08 ─────────────────────────────────
nombres_clases = [f"GG {c}" for c in le.classes_]

reporte = classification_report(
    etiquetas_reales,
    preds,
    target_names=nombres_clases,
    output_dict=True,
    zero_division=0
)

filas = []
for nombre in nombres_clases:
    m = reporte[nombre]
    filas.append({
        "Clase":     nombre,
        "Precisión": round(m["precision"], 3),
        "Recall":    round(m["recall"], 3),
        "F1-score":  round(m["f1-score"], 3),
        "N":         int(m["support"]),
    })

for etiqueta, clave in [("Macro (promedio)", "macro avg"), ("Weighted (promedio)", "weighted avg")]:
    m = reporte[clave]
    filas.append({
        "Clase":     etiqueta,
        "Precisión": round(m["precision"], 3),
        "Recall":    round(m["recall"], 3),
        "F1-score":  round(m["f1-score"], 3),
        "N":         int(m["support"]),
    })

tabla = pd.DataFrame(filas)
separador = pd.DataFrame([{"Clase": "-" * 20, "Precisión": "-" * 9, "Recall": "-" * 6, "F1-score": "-" * 8, "N": "-" * 4}])
tabla = pd.concat([tabla.iloc[:-2], separador, tabla.iloc[-2:]], ignore_index=True)

print("\nRendimiento del modelo por gran grupo CIUO-08 (conjunto de test)")
print("=" * 62)
print(tabla.to_string(index=False))
print(f"\nExactitud global: {reporte['accuracy']:.1%}")

/home/jovyan/work/conda_envs/capacitacion/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Rendimiento del modelo por gran grupo CIUO-08 (conjunto de test)
               Clase Precisión Recall F1-score    N
              GG -99       0.0    0.0      0.0    4
                GG 0     0.545  0.667      0.6   36
                GG 1     0.672  0.667    0.669  129
                GG 2     0.826  0.787    0.806  338
                GG 3     0.577  0.479    0.524  219
                GG 4      0.63  0.708    0.667  130
                GG 5     0.749  0.742    0.745  302
                GG 6     0.737  0.712    0.724  118
                GG 7     0.691   0.79    0.737  252
                GG 8     0.789  0.829    0.808  140
                GG 9      0.71  0.704    0.707  233
              GG 999     0.471  0.471    0.471  138
-------------------- --------- ------ -------- ----
    Macro (promedio)     0.616   0.63    0.622 2039
 Weighted (promedio)     0.697  0.699    0.697 2039

Exactitud global: 69.9%


## 9. Guardado del modelo

Guardamos el modelo entrenado y el tokenizador en el mismo directorio. Para inferencia futura basta cargar ambos con `from_pretrained()` apuntando a esa carpeta local.

In [9]:
DIRECTORIO_FINAL = "./modelo_ciuo08/final"

trainer.save_model(DIRECTORIO_FINAL)
tokenizer.save_pretrained(DIRECTORIO_FINAL)

print(f"Modelo y tokenizador guardados en: {DIRECTORIO_FINAL}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.15it/s]

Modelo y tokenizador guardados en: ./modelo_ciuo08/final


## 10. Inferencia sobre nuevas glosas

Para usar el modelo en producción se carga desde la carpeta guardada usando `pipeline`, que internamente maneja la tokenización y la conversión de logits a predicciones.

In [ ]:
from transformers import pipeline

clasificador = pipeline(
    "text-classification",
    model=DIRECTORIO_FINAL,
    tokenizer=DIRECTORIO_FINAL,
    device=0   # GPU si está disponible; cambiar a -1 para CPU
)

# El pipeline devuelve "LABEL_0", "LABEL_1", etc.
# Recuperamos el código CIUO-08 original con el LabelEncoder
def predecir_ciuo(texto):
    resultado = clasificador(texto, truncation=True, max_length=64)[0]
    idx       = int(resultado["label"].split("_")[1])
    codigo    = le.classes_[idx]
    return {"codigo_ciuo_1d": int(codigo), "confianza": round(resultado["score"], 4)}

In [27]:
glosas = [
    "OCUPACIÓN: ENFERMERA; TAREA: ADMINISTRAR MEDICAMENTOS Y CONTROLAR SIGNOS VITALES",
    "OCUPACIÓN: CONDUCTOR DE CAMIÓN; TAREA: TRANSPORTAR CARGA ENTRE REGIONES",
    "OCUPACIÓN: TEMPORERO AGRÍCOLA; TAREA: COSECHAR UVA EN TEMPORADA",
    "OCUPACIÓN: MIMO; TAREA: HACER MÍMICAS Y ENTRETENER"
]

for glosa in glosas:
    r = predecir_ciuo(glosa)
    print(f"GG {r['codigo_ciuo_1d']}  (confianza {r['confianza']:.1%})  →  {glosa[:65]}...")

GG 2  (confianza 35.5%)  →  OCUPACIÓN: ENFERMERA; TAREA: ADMINISTRAR MEDICAMENTOS Y CONTROLAR...
GG 8  (confianza 96.2%)  →  OCUPACIÓN: CONDUCTOR DE CAMIÓN; TAREA: TRANSPORTAR CARGA ENTRE RE...
GG 9  (confianza 93.7%)  →  OCUPACIÓN: TEMPORERO AGRÍCOLA; TAREA: COSECHAR UVA EN TEMPORADA...
GG 7  (confianza 63.8%)  →  OCUPACIÓN: MIMO; TAREA: HACER MÍMICAS Y ENTRETENER...
